# 06 – Interactive Visual Analytics with Folium

**Goal:** put every launch site and every launch on a map, then measure how close a launch site is to the coast, a railway, a highway and a city.

**Data:** `spacex_launch_geo.csv` from the IBM course (56 launches with site coordinates and `class`).
**Proximity data:** OpenStreetMap, queried with the Overpass API (real map features, no hand-typed coordinates).

Maps are saved as HTML files in `images/` (open them in a browser). Screenshots are taken in notebook step 6.

In [1]:
import os, json, math, time
import requests
import pandas as pd
import folium
from folium.plugins import MarkerCluster, MousePosition
from folium.features import DivIcon

IMG = "../images/"
CSV = "../data/spacex_launch_geo.csv"
if not os.path.exists(CSV):
    url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/spacex_launch_geo.csv"
    pd.read_csv(url).to_csv(CSV, index=False)
spacex_df = pd.read_csv(CSV)[["Launch Site", "Lat", "Long", "class"]]
print(spacex_df.shape)
spacex_df.head()

(56, 4)


,Launch Site,Lat,Long,class
0,CCAFS LC-40,28.562302,-80.577356,0
1,CCAFS LC-40,28.562302,-80.577356,0
2,CCAFS LC-40,28.562302,-80.577356,0
3,CCAFS LC-40,28.562302,-80.577356,0
4,CCAFS LC-40,28.562302,-80.577356,0


## 1. Mark all launch sites (marker + circle)

In [2]:
launch_sites_df = spacex_df.groupby("Launch Site", as_index=False).first()[["Launch Site", "Lat", "Long"]]
launch_sites_df

,Launch Site,Lat,Long
0,CCAFS LC-40,28.562302,-80.577356
1,CCAFS SLC-40,28.563197,-80.576820
2,KSC LC-39A,28.573255,-80.646895
3,VAFB SLC-4E,34.632834,-120.610745


In [3]:
NASA_JSC = [29.559684888503615, -95.0830971930759]       # start the map at NASA Johnson Space Center
site_map = folium.Map(location=NASA_JSC, zoom_start=4)

for _, s in launch_sites_df.iterrows():
    coord = [s["Lat"], s["Long"]]
    folium.Circle(coord, radius=1000, color="#d35400", fill=True, popup=s["Launch Site"]).add_to(site_map)
    folium.Marker(coord, icon=DivIcon(icon_size=(20, 20), icon_anchor=(0, 0),
        html=f'<div style="font-size:18px;color:#d35400;font-weight:bold;white-space:nowrap">{s["Launch Site"]}</div>')).add_to(site_map)

site_map.save(IMG + "folium_1_launch_sites.html")
site_map

**Finding:** all launch sites are in the south of the USA and very close to the coast –
three in Florida (Cape Canaveral, east coast) and one in California (Vandenberg, west coast).

## 2. Mark every launch: green = landed, red = not landed (MarkerCluster)

In [4]:
spacex_df["marker_color"] = spacex_df["class"].map({1: "green", 0: "red"})

launch_map = folium.Map(location=[28.5, -80.6], zoom_start=10)
marker_cluster = MarkerCluster().add_to(launch_map)
for _, r in spacex_df.iterrows():
    folium.Marker([r["Lat"], r["Long"]],
                  icon=folium.Icon(color="white", icon_color=r["marker_color"]),
                  popup=f'{r["Launch Site"]} – class {r["class"]}').add_to(marker_cluster)

launch_map.save(IMG + "folium_2_launch_outcomes.html")
launch_map

In [5]:
site_rate = spacex_df.groupby("Launch Site")["class"].agg(launches="count", landed="sum")
site_rate["success_rate_%"] = (site_rate["landed"] / site_rate["launches"] * 100).round(1)
site_rate.sort_values("success_rate_%", ascending=False)

,launches,landed,success_rate_%
Launch Site,,,
KSC LC-39A,13,10,76.9
CCAFS SLC-40,7,3,42.9
VAFB SLC-4E,10,4,40.0
CCAFS LC-40,26,7,26.9


## 3. Proximity analysis with OpenStreetMap

In [6]:
OVERPASS = "https://overpass-api.de/api/interpreter"
CACHE = "../data/raw/osm_cache.json"
cache = json.load(open(CACHE)) if os.path.exists(CACHE) else {}

def overpass(query):
    """Run an Overpass query (cached in data/raw so the notebook can be re-run offline)."""
    if query not in cache:
        for attempt in range(5):                 # the free server is sometimes busy: try again
            r = requests.post(OVERPASS, data={"data": query}, headers={"User-Agent": "spacex-capstone-student/1.0"}, timeout=180)
            if r.status_code == 200:
                break
            time.sleep(15)
        r.raise_for_status()
        cache[query] = r.json()
        json.dump(cache, open(CACHE, "w"))
        time.sleep(2)                        # be polite to the free server
    return cache[query]

def calculate_distance(lat1, lon1, lat2, lon2):
    """Distance in km between two points (haversine formula)."""
    R = 6373.0
    lat1, lon1, lat2, lon2 = map(math.radians, [lat1, lon1, lat2, lon2])
    a = math.sin((lat2 - lat1) / 2) ** 2 + math.cos(lat1) * math.cos(lat2) * math.sin((lon2 - lon1) / 2) ** 2
    return R * 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))

# what we look for around each site: (Overpass filter, search radius in metres)
TARGETS = {
    "coastline": ('way["natural"="coastline"]', 15000),
    "railway":   ('way["railway"="rail"]', 30000),
    "highway":   ('way["highway"~"^(motorway|trunk|primary)$"]', 30000),
    "city":      ('node["place"="city"]', 100000),
}

def nearest(lat, lon, kind):
    """Return (distance_km, lat, lon, name) of the nearest map feature of this kind."""
    flt, radius = TARGETS[kind]
    data = overpass(f"[out:json][timeout:120];{flt}(around:{radius},{lat},{lon});out geom;")
    best = None
    for el in data["elements"]:
        points = el.get("geometry") or [{"lat": el["lat"], "lon": el["lon"]}]
        for p in points:                                   # check every point of the line
            d = calculate_distance(lat, lon, p["lat"], p["lon"])
            if best is None or d < best[0]:
                best = (d, p["lat"], p["lon"], el.get("tags", {}).get("name", kind))
    return best

In [7]:
rows = []
for _, s in launch_sites_df.iterrows():
    for kind in TARGETS:
        d, lat, lon, name = nearest(s["Lat"], s["Long"], kind)
        rows.append({"Launch Site": s["Launch Site"], "feature": kind, "name": name,
                     "distance_km": round(d, 2), "lat": lat, "lon": lon})
prox = pd.DataFrame(rows)
print(prox.pivot(index="Launch Site", columns="feature", values="distance_km")[list(TARGETS)])

feature       coastline  railway  highway   city
Launch Site                                     
CCAFS LC-40        0.95     0.05    11.07  23.21
CCAFS SLC-40       0.87     0.02    11.17  23.23
KSC LC-39A         6.45     0.69     6.89  16.33
VAFB SLC-4E        1.34     0.01    12.39  39.04


In [8]:
prox[["Launch Site", "feature", "name", "distance_km"]]

,Launch Site,feature,name,distance_km
0,CCAFS LC-40,coastline,coastline,0.95
1,CCAFS LC-40,railway,railway,0.05
2,CCAFS LC-40,highway,Space Commerce Way,11.07
3,CCAFS LC-40,city,Titusville,23.21
4,CCAFS SLC-40,coastline,coastline,0.87
5,CCAFS SLC-40,railway,railway,0.02
6,CCAFS SLC-40,highway,Space Commerce Way,11.17
7,CCAFS SLC-40,city,Titusville,23.23
8,KSC LC-39A,coastline,coastline,6.45
9,KSC LC-39A,railway,railway,0.69


## 4. Draw distance lines from CCAFS SLC-40 (with MousePosition to read coordinates)

In [9]:
SITE = "CCAFS SLC-40"
s = launch_sites_df[launch_sites_df["Launch Site"] == SITE].iloc[0]
site_coord = [s["Lat"], s["Long"]]
COLORS = {"coastline": "#1f77b4", "railway": "#6a3d9a", "highway": "#e67e22", "city": "#c0392b"}

prox_map = folium.Map(location=site_coord, zoom_start=11)
MousePosition(position="topright", separator=" Long: ", prefix="Lat:", num_digits=5).add_to(prox_map)
folium.Marker(site_coord, popup=SITE, icon=folium.Icon(color="blue", icon="rocket", prefix="fa")).add_to(prox_map)

for _, p in prox[prox["Launch Site"] == SITE].iterrows():
    target = [p["lat"], p["lon"]]
    folium.PolyLine([site_coord, target], color=COLORS[p["feature"]], weight=3).add_to(prox_map)
    folium.CircleMarker(target, radius=5, color=COLORS[p["feature"]], fill=True).add_to(prox_map)
    folium.Marker(target, icon=DivIcon(icon_size=(0, 0), icon_anchor=(-8, 10),
        html=f'<div style="display:inline-block;font-size:15px;font-weight:bold;color:{COLORS[p["feature"]]};'
             f'background:white;padding:2px 6px;border:2px solid {COLORS[p["feature"]]};border-radius:4px;white-space:nowrap">'
             f'{p["feature"]}: {p["distance_km"]:.2f} km</div>')).add_to(prox_map)

prox_map.save(IMG + "folium_3_proximity.html")
prox_map

## 5. Findings
* **Coast:** every site is very close to the ocean, so a failed rocket falls into the sea, not on people.
* **Railway / highway:** sites are near transport lines to bring in heavy rocket parts.
* **City:** cities are kept far away for safety.

(The exact distances are in the table above.)

## 6. Screenshots of the maps (headless Chromium via Playwright)

In [10]:
import sys, asyncio
from concurrent.futures import ThreadPoolExecutor
from playwright.sync_api import sync_playwright

def shot(page, html, png, wait=6000, clicks=None):
    page.goto("file:///" + os.path.abspath(IMG + html).replace("\\", "/"))
    page.wait_for_timeout(wait)
    for c in (clicks or []):
        c(page); page.wait_for_timeout(5000)          # give the map tiles time to load
    page.screenshot(path=IMG + png)
    print("saved", png)

def open_cluster(page):
    page.locator(".marker-cluster").first.click()

def zoom_to(lat, lon, zoom):
    js = f"() => {{ const m = Object.values(window).find(v => v instanceof L.Map); m.setView([{lat}, {lon}], {zoom}); }}"
    return lambda page: page.evaluate(js)

ksc = launch_sites_df[launch_sites_df["Launch Site"] == "KSC LC-39A"].iloc[0]

def take_screenshots():
    # on Windows, Playwright needs the "Proactor" event loop to start the browser
    if sys.platform == "win32":
        asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())
    with sync_playwright() as p:
        browser = p.chromium.launch()
        page = browser.new_page(viewport={"width": 1600, "height": 900})
        shot(page, "folium_1_launch_sites.html", "folium_1_launch_sites.png", clicks=[zoom_to(32, -100, 5)])
        shot(page, "folium_1_launch_sites.html", "folium_1b_launch_sites_florida.png", clicks=[zoom_to(28.585, -80.60, 12)])
        shot(page, "folium_2_launch_outcomes.html", "folium_2_launch_outcomes.png")
        # zoom into KSC LC-39A and open its cluster to show the green / red markers
        shot(page, "folium_2_launch_outcomes.html", "folium_2b_launch_outcomes_ksc.png",
             clicks=[zoom_to(ksc["Lat"], ksc["Long"], 17), open_cluster])
        # proximity: overview (highway, city) and close-up (coastline, railway)
        shot(page, "folium_3_proximity.html", "folium_3_proximity.png",
             clicks=[zoom_to(site_coord[0] + 0.03, site_coord[1] - 0.12, 11)])
        shot(page, "folium_3_proximity.html", "folium_3b_proximity_closeup.png",
             clicks=[zoom_to(site_coord[0], site_coord[1] + 0.004, 15)])
        browser.close()

# Playwright's sync API cannot run inside Jupyter's event loop, so we run it in a separate thread
ThreadPoolExecutor(1).submit(take_screenshots).result()

saved folium_1_launch_sites.png


saved folium_1b_launch_sites_florida.png


saved folium_2_launch_outcomes.png


saved folium_2b_launch_outcomes_ksc.png


saved folium_3_proximity.png


saved folium_3b_proximity_closeup.png
